#  GenAI Career Assistant – Multi-Agent AI Mentor


This notebook implements an AI-powered multi-agent workflow using LangChain, LangGraph, Gemini, and DuckDuckGo Search to assist users with:
- GenAI learning guidance  
- Resume support  
- Interview preparation  
- Job search assistance  

The notebook includes:
1. Query classification  
2. Sub-category routing  
3. Content generation (blogs, tutorials, mock interviews, resumes)  
4. Workflow graph execution 

### Before starting please install the below packages

In [ ]:
pip install langchain==0.3.7 langchain-community==0.3.7 langchain_google_genai==2.0.4 duckduckgo_search==6.3.4 langgraph==0.2.48 python-dotenv

### Here we will import necessary packages:
`langgraph`, `langchain_core`, `langchain_google_genai` - These are important packages for our project.
 

In [ ]:
from typing import Dict, TypedDict
from langgraph.graph import StateGraph, END, START  
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

from IPython.display import display, Image, Markdown
from langchain_core.runnables.graph import MermaidDrawMethod  
from dotenv import load_dotenv
import os

 
load_dotenv()

# Set the Gemini API key for authentication with Google Generative AI services
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# Instantiate a chat model using Google's Gemini-2.5-flash with specified configurations
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",
                             verbose=True,
                             temperature=0.5,
                             google_api_key=os.getenv("GOOGLE_API_KEY"))


### Defining a State class using TypedDict to specify the structure of state data in the workflow.
- `query`: a string representing the user's input or question.
- `category`: a string indicating the category or type of the query.
- `response`: a string holding the AI model's generated response to the query.
This TypedDict ensures each state has a consistent data format for easier management and readability.

In [4]:
class State(TypedDict):
    query: str
    category: str
    response: str

### First we are defining utilities we will require further
<a href="https://python.langchain.com/docs/how_to/trim_messages/"> <a>

1. **`trim_conversation` Function**: This function limits the conversation history to the latest messages (up to 10), ensuring only recent and relevant messages are retained in the promp
  
2. **`save_file` Function**: Saves data into a uniquely timestamped Markdown file in the `Agent_output` folder, creating the folder if it doesn't exst.

3. **`show_md_file` Function**: Reads and displays the content of a Markdown file within the notebook, rendering it in Markdown form readabilityblity.
lity.


In [ ]:
# Importing message types and utilities from langchain_core:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, trim_messages

def trim_conversation(prompt):
    """Trims conversation history to retain only the latest messages within the limit."""
    max_messages = 10 
    return trim_messages(
        prompt,
        max_tokens=max_messages,   
        strategy="last",  
        token_counter=len,   
        start_on="human",   
        include_system=True,   
        allow_partial=False,   
    )

import os
from datetime import datetime

def save_file(data, filename):
    """Saves data to a markdown file with a timestamped filename."""
    folder_name = "Agent_output"  
    os.makedirs(folder_name, exist_ok=True)   
    
    # Generate a timestamped filename for uniqueness
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")  
    filename = f"{filename}_{timestamp}.md"
    
    
    file_path = os.path.join(folder_name, filename)
    
    # Save the data to the file in the specified path
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(data)
        print(f"File '{file_path}' created successfully.")
    
     
    return file_path

def show_md_file(file_path):
    """Displays the content of a markdown file as Markdown in the notebook."""
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    
    display(Markdown(content))


### Learning Resource Agent (Tutorial & Q&A)
Uses LangChain to create a chat-based AI assistant for learning.

### Imports: ChatPromptTemplate, MessagesPlaceholder, DuckDuckGoSearchResults, create_tool_calling_agent, AgentExecutor.
Class Features:
Init: Sets up chat model (gemini-2.5-flash), prompt, and tools.
TutorialAgent: Searches online tutorials and saves outputs as Markdown.
QueryBot: Interactive Q&A session; trims conversation as it grows, type 'exit' to end.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.tools import DuckDuckGoSearchResults  
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor

class LearningResourceAgent:
    def __init__(self, prompt):
        # Initialize the chat model, prompt template, and search tools
        self.model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
        self.prompt = prompt
        self.tools = [DuckDuckGoSearchResults()]

    def TutorialAgent(self, user_input):
        # Set up an agent with tool access and execute a tutorial-style response
        agent = create_tool_calling_agent(self.model, self.tools, self.prompt)
        agent_executor = AgentExecutor(agent=agent, tools=self.tools, verbose=True)
        response = agent_executor.invoke({"input": user_input})
        
        # Save and display the response as a markdown file
        path = save_file(str(response.get('output')).replace("```markdown", "").strip(), 'Tutorial')
        print(f"Tutorial saved to {path}")
        return path

    def QueryBot(self, user_input):
        # Initiates a Q&A loop for continuous interaction with the user
        print("\nStarting the Q&A session. Type 'exit' to end the session.\n")
        record_QA_session = []
        record_QA_session.append('User Query: %s \n' % user_input)
        self.prompt.append(HumanMessage(content=user_input))
        while True:
            self.prompt = trim_conversation(self.prompt)
            
            # Generate a response from the AI model and update conversation history
            response = self.model.invoke(self.prompt)
            record_QA_session.append('\nExpert Response: %s \n' % response.content)
            
            self.prompt.append(AIMessage(content=response.content))
            
            # Display the AI's response and prompt for user input
            print('*' * 50 + 'AGENT' + '*' * 50)
            print("\nEXPERT AGENT RESPONSE:", response.content)
            
            print('*' * 50 + 'USER' + '*' * 50)
            user_input = input("\nYOUR QUERY: ")
            record_QA_session.append('\nUser Query: %s \n' % response.content)
            self.prompt.append(HumanMessage(content=user_input))
            
            # Exit the Q&A loop if the user types 'exit'
            if user_input.lower() == "exit":
                print("Ending the chat session.")
                path = save_file(''.join(record_QA_session),'Q&A_Doubt_Session')
                print(f"Q&A Session saved to {path}")
                return path


### Interview Handling Agent (Prep & Mock Interview)

Class: InterviewAgent

Init: Sets up chat model (gemini-1.5-flash), prompt, tools, and agent executor with error handling.

Interview_questions: Handles user questions, stores responses in questions_bank, saves chat as Markdown, ends with 'exit'.

Mock_Interview: Simulates an interview; shows model as "Interviewer" and user as "Candidate", trims conversation dynamically, ends with 'exit', returns full session record.


In [ ]:
class InterviewAgent:
    def __init__(self, prompt):
        # Initialize the chat model, prompt template, and search tool for use in the agent
        self.model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
        self.prompt = prompt
        self.tools = [DuckDuckGoSearchResults()]  

    def Interview_questions(self, user_input):
        chat_history = []
        questions_bank = ''
        # Create an agent executor with tool access and enable verbose output and error handling
        self.agent = create_tool_calling_agent(self.model, self.tools, self.prompt)
        self.agent_executor = AgentExecutor(agent=self.agent, tools=self.tools, verbose=True, handle_parsing_errors=True)
        while True:
            print("\nStarting the Interview question preparation. Type 'exit' to end the session.\n")
            if user_input.lower() == "exit":
                print("Ending the conversation. Goodbye!")
                break
            
            # Generate a response to the user input and add it to questions_bank
            response = self.agent_executor.invoke({"input": user_input, "chat_history": chat_history})
            questions_bank += str(response.get('output')).replace("```markdown", "").strip() + "\n"
            
            chat_history.extend([HumanMessage(content=user_input), response["output"]])
            if len(chat_history) > 10:
                chat_history = chat_history[-10:]   
            
            user_input = input("You: ")
        
        # Save the entire question-response history to a markdown file and display it
        path = save_file(questions_bank, 'Interview_questions')
        print(f"Interviews question saved to {path}")
        return path

    def Mock_Interview(self):
        print("\nStarting the mock interview. Type 'exit' to end the session.\n")
        
        # Initialize with a starting message and store interview records
        initial_message = 'I am ready for the interview.\n'
        interview_record = []
        interview_record.append('Candidate: %s \n' % initial_message)
        self.prompt.append(HumanMessage(content=initial_message))
        
        while True:
            self.prompt = trim_conversation(self.prompt)
            response = self.model.invoke(self.prompt)
            self.prompt.append(AIMessage(content=response.content))
            
            # Output the AI's response as the "Interviewer"
            print("\nInterviewer:", response.content)
            interview_record.append('\nInterviewer: %s \n' % response.content)
            
            # Get the user's response as "Candidate" input
            user_input = input("\nCandidate: ")
            interview_record.append('\nCandidate: %s \n' % user_input)
            
            self.prompt.append(HumanMessage(content=user_input))
            
            # End the interview if the user types "exit"
            if user_input.lower() == "exit":
                print("Ending the interview session.")
                path = save_file(''.join(interview_record),'Mock_Interview')
                print(f"Mock Interview saved to {path}")
                return path


### Resume Creation Agent

Class: ResumeMaker

Init: Sets up chat model (gemini-2.5-flash), prompt, tools, and agent executor with error handling.

Create_Resume: Engages in a conversational loop to gather user input, generates resume content, maintains last 10 messages for context, ends with 'exit', and saves the final resume as a timestamped Markdown file.


In [ ]:
class ResumeMaker:
    def __init__(self, prompt):
        self.model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
        self.prompt = prompt
        self.tools = [DuckDuckGoSearchResults()]   
        # Create an agent executor with tool access, enabling verbose output and error handling
        self.agent = create_tool_calling_agent(self.model, self.tools, self.prompt)
        self.agent_executor = AgentExecutor(agent=self.agent, tools=self.tools, verbose=True, handle_parsing_errors=True)

    def Create_Resume(self, user_input):
        # Maintain chat history for the resume creation conversation
        chat_history = []
        while True:
            print("\nStarting the Resume create session. Type 'exit' to end the session.\n")
            if user_input.lower() == "exit":
                print("Ending the conversation. Goodbye!")
                break
            
            # Generate a response to user input using the agent and add it to the chat history
            response = self.agent_executor.invoke({"input": user_input, "chat_history": chat_history})
            chat_history.extend([HumanMessage(content=user_input), response["output"]])
             
            if len(chat_history) > 10:
                chat_history = chat_history[-10:]
             
            user_input = input("You: ")
        
        # Save the final output as a markdown file and return the file path
        path = save_file(str(response.get('output')).replace("```markdown", "").strip(), 'Resume')
        print(f"Resume saved to {path}")
        return path


### Job Search Agent

Class: JobSearch

Init: Sets up chat model (gemini-2.5-pro), prompt, search tools, and agent executor with verbose output and error handling.

find_jobs: Engages users in a conversational job search loop, keeps last 10 messages for context, ends with 'exit', and saves the conversation as a timestamped Markdown file.


In [ ]:
class JobSearch:
    def __init__(self, prompt):
        # Initialize the chat model, prompt template, and search tool for job search assistance
        self.model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
        self.prompt = prompt
        self.tools = DuckDuckGoSearchResults()   
      

    def find_jobs(self, user_input):
        results = self.tools.invoke(user_input)
        chain = self.prompt | self.model  
        jobs = chain.invoke({"result": results}).content
        
        path = save_file(str(jobs).replace("```markdown", "").strip(), 'Job_search')
        print(f"Jobs saved to {path}")
        return path


### User Query Categorization
Purpose: Helps the AI understand the user’s intent and route queries to the appropriate module.

categorize: Classifies user input into Learning AI, Resume, Interview, or Job Search using few-shot examples.

handle_learning_resource: Distinguishes tutorial vs general AI questions.

handle_interview_preparation: Identifies mock interview vs general interview queries.


In [23]:
def categorize(state: State) -> State:
    """Categorizes the user query into one of four main categories: Learn Generative AI Technology, Resume Making, Interview Preparation, or Job Search."""
    prompt = ChatPromptTemplate.from_template(
        "Categorize the following customer query into one of these categories:\n"
        "1: Learn Generative AI Technology\n"
        "2: Resume Making\n"
        "3: Interview Preparation\n"
        "4: Job Search\n"
        "Give the number only as an output.\n\n"
        "Examples:\n"
        "1. Query: 'What are the basics of generative AI, and how can I start learning it?' -> 1\n"
        "2. Query: 'Can you help me improve my resume for a tech position?' -> 2\n"
        "3. Query: 'What are some common questions asked in AI interviews?' -> 3\n"
        "4. Query: 'Are there any job openings for AI engineers?' -> 4\n\n"
        "Now, categorize the following customer query:\n"
        "Query: {query}"
    )

    # Creates a categorization chain and invokes it with the user's query to get the category
    chain = prompt | llm 
    print('Categorizing the customer query...')
    category = chain.invoke({"query": state["query"]}).content
    return {"category": category}

def handle_learning_resource(state: State) -> State:
    """Determines if the query is related to Tutorial creation or general Questions on generative AI topics."""
    prompt = ChatPromptTemplate.from_template(
        "Categorize the following user query into one of these categories:\n\n"
        "Categories:\n"
        "- Tutorial: For queries related to creating tutorials, blogs, or documentation on generative AI.\n"
        "- Question: For general queries asking about generative AI topics.\n"
        "- Default to Question if the query doesn't fit either of these categories.\n\n"
        "Examples:\n"
        "1. User query: 'How to create a blog on prompt engineering for generative AI?' -> Category: Tutorial\n"
        "2. User query: 'Can you provide a step-by-step guide on fine-tuning a generative model?' -> Category: Tutorial\n"
        "3. User query: 'Provide me the documentation for Langchain?' -> Category: Tutorial\n"
        "4. User query: 'What are the main applications of generative AI?' -> Category: Question\n"
        "5. User query: 'Is there any generative AI course available?' -> Category: Question\n\n"
        "Now, categorize the following user query:\n"
        "The user query is: {query}\n"
    )

    # Creates a further categorization chain to decide between Tutorial or Question
    chain = prompt | llm 
    print('Categorizing the customer query further...')
    response = chain.invoke({"query": state["query"]}).content
    return {"category": response}

def handle_interview_preparation(state: State) -> State:
    """Determines if the query is related to Mock Interviews or general Interview Questions."""
    prompt = ChatPromptTemplate.from_template(
        "Categorize the following user query into one of these categories:\n\n"
        "Categories:\n"
        "- Mock: For requests related to mock interviews.\n"
        "- Question: For general queries asking about interview topics or preparation.\n"
        "- Default to Question if the query doesn't fit either of these categories.\n\n"
        "Examples:\n"
        "1. User query: 'Can you conduct a mock interview with me for a Gen AI role?' -> Category: Mock\n"
        "2. User query: 'What topics should I prepare for an AI Engineer interview?' -> Category: Question\n"
        "3. User query: 'I need to practice interview focused on Gen AI.' -> Category: Mock\n"
        "4. User query: 'Can you list important coding topics for AI tech interviews?' -> Category: Question\n\n"
        "Now, categorize the following user query:\n"
        "The user query is: {query}\n"
    )

    # Creates a further categorization chain to decide between Mock or Question
    chain = prompt | llm 
    print('Categorizing the customer query further...')
    response = chain.invoke({"query": state["query"]}).content
    return {"category": response}


### Job Search & Resume Functions

job_search: Uses an AI agent to find Generative AI job listings and generates a Markdown file with company, role, and link details.

handle_resume_making: Collects user info (skills, experience, projects) to create a tailored AI-focused resume in Markdown format.


In [26]:
def job_search(state: State) -> State:
    """Provide a job search response based on user query requirements."""
    prompt = ChatPromptTemplate.from_template('''Your task is to refactor and make .md file for the this content which includes
    the jobs available in the market. Refactor such that user can refer easily. Content: {result}''')
    jobSearch = JobSearch(prompt)
    state["query"] = input('Please make sure to mention Job location you want,Job roles\n')
    path = jobSearch.find_jobs(state["query"])
    show_md_file(path)
    return {"response": path}

def handle_resume_making(state: State) -> State:
    """Generate a customized resume based on user details for a tech role in AI and Generative AI."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", '''You are a skilled resume expert with extensive experience in crafting resumes tailored for tech roles, especially in AI and Generative AI. 
        Your task is to create a resume template for an AI Engineer specializing in Generative AI, incorporating trending keywords and technologies in the current job market. 
        Feel free to ask users for any necessary details such as skills, experience, or projects to complete the resume. 
        Try to ask details step by step and try to ask all details within 4 to 5 steps.
        Ensure the final resume is in .md format.'''),
       MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ])
    resumeMaker = ResumeMaker(prompt)
    path = resumeMaker.Create_Resume(state["query"])
    show_md_file(path)
    return {"response": path}


### Next we will create a function for Q&A query bot and Tutorial maker

3. **`ask_query_bot` Function**:
   - Engages in a conversational Q&A session, providing detailed answers to user queries related to Generative AI.
   - Uses back-and-forth interaction to ensure clarity and completeness in responses.

4. **`tutorial_agent` Function**:
   - Generates comprehensive tutorial blogs on Generative AI topics with explanations, example code, and resources for further learning.
   - Saves the tutorial in Markdown (.md) format, designed for clarity and learning support.

In [ ]:
def ask_query_bot(state: State) -> State:
    """Provide detailed answers to user queries related to Generative AI."""
    system_message = '''You are an expert Generative AI Engineer with extensive experience in training and guiding others in AI engineering. 
    You have a strong track record of solving complex problems and addressing various challenges in AI. 
    Your role is to assist users by providing insightful solutions and expert advice on their queries.
    Engage in a back-and-forth chat session to address user queries.'''
    prompt = [SystemMessage(content=system_message)]

    learning_agent = LearningResourceAgent(prompt)

    path = learning_agent.QueryBot(state["query"])
    show_md_file(path)
    return {"response": path}

def tutorial_agent(state: State) -> State:
    """Generate a tutorial blog for Generative AI based on user requirements."""
    system_message = '''You are a knowledgeable assistant specializing as a Senior Generative AI Developer with extensive experience in both development and tutoring. 
         Additionally, you are an experienced blogger who creates tutorials focused on Generative AI.
         Your task is to develop high-quality tutorials blogs in .md file with Coding example based on the user's requirements. 
         Ensure tutorial includes clear explanations, well-structured python code, comments, and fully functional code examples.
         Provide resource reference links at the end of each tutorial for further learning.'''
    prompt = ChatPromptTemplate.from_messages([("system", system_message),
            ("placeholder", "{chat_history}"),
            ("human", "{input}"),
            ("placeholder", "{agent_scratchpad}"),])
     learning_agent = LearningResourceAgent(prompt)
    path = learning_agent.TutorialAgent(state["query"])
    show_md_file(path)
    return {"response": path}



### Finally we will create function Interview Question prep and Mock interview

5. **`interview_topics_quesions` Function**:
   - Provides a list of interview questions tailored to Generative AI roles, along with references where possible.
   - Outputs a Markdown (.md) document with curated questions based on user requirements.

6. **`mock_interview` Function**:
   - Conducts a simulated Generative AI job interview, interacting with the user in real-time.
   - Provides a post-interview evaluation, summarizing the candidate's performance.

In [32]:
def interview_topics_questions(state: State) -> State:
    """Provide a curated list of interview questions related to Generative AI based on user input."""
    system_message = '''You are a good researcher in finding interview questions for Generative AI topics and jobs.
                     Your task is to provide a list of interview questions for Generative AI topics and job based on user requirements.
                     Provide top questions with references and links if possible. You may ask for clarification if needed.
                     Generate a .md document containing the questions.'''
    prompt = ChatPromptTemplate.from_messages([
                        ("system", system_message),
                        MessagesPlaceholder("chat_history"),
                        ("human", "{input}"),
                        ("placeholder", "{agent_scratchpad}"),])
    interview_agent = InterviewAgent(prompt)
    path = interview_agent.Interview_questions(state["query"])
    show_md_file(path)
    return {"response": path}

def mock_interview(state: State) -> State:
    """Conduct a mock interview for a Generative AI position, including evaluation at the end."""
    system_message = '''You are a Generative AI Interviewer. You have conducted numerous interviews for Generative AI roles.
         Your task is to conduct a mock interview for a Generative AI position, engaging in a back-and-forth interview session.
         The conversation should not exceed more than 15 to 20 minutes.
         At the end of the interview, provide an evaluation for the candidate.'''
    prompt = [SystemMessage(content=system_message)]
    interview_agent = InterviewAgent(prompt)
    path = interview_agent.Mock_Interview()
    show_md_file(path)
    return {"response": path}

### Query Routing Functions

Purpose: Directs user queries to the correct handler after categorization.

route_query: Sends queries to Learning, Resume, Interview, or Job Search handlers; prompts user if unclear.

route_interview: Routes interview queries to Mock Interview or general Interview Topics; defaults to mock if uncertain.

route_learning: Routes learning queries to Tutorial creation or Q&A; returns False if unclear.


In [ ]:
def route_query(state: State):
    """Route the query based on its category to the appropriate handler."""
    if '1' in state["category"]:
        print('Category: handle_learning_resource')
        return "handle_learning_resource"   
    elif '2' in state["category"]:
        print('Category: handle_resume_making')
        return "handle_resume_making"  
    elif '3' in state["category"]:
        print('Category: handle_interview_preparation')
        return "handle_interview_preparation"   
    elif '4' in state["category"]:
        print('Category: job_search')
        return "job_search"  
    else:
        print("Please ask your question based on my description.")
        return False   

def route_interview(state: State) -> str:
    """Route the query to the appropriate interview-related handler."""
    if 'Question'.lower() in state["category"].lower():
        print('Category: interview_topics_questions')
        return "interview_topics_questions"   
    elif 'Mock'.lower() in state["category"].lower():
        print('Category: mock_interview')
        return "mock_interview"  
    else:
        print('Category: mock_interview')
        return "mock_interview"  

def route_learning(state: State):
    """Route the query based on the learning path category."""
    if 'Question'.lower() in state["category"].lower():
        print('Category: ask_query_bot')
        return "ask_query_bot" 
    elif 'Tutorial'.lower() in state["category"].lower():
        print('Category: tutorial_agent')
        return "tutorial_agent"  
    else:
        print("Please ask your question based on my interview description.")
        return False   


### Now all set lets create workflow graphs adding edges and nodes.

Purpose: Builds a query-handling workflow using StateGraph.

Nodes & Edges: Defines nodes for each category handler (categorize, handle_learning_resource, handle_resume_making, handle_interview_preparation, job_search) and connects them with conditional edges for routing (route_query, route_learning, route_interview).

Endpoints: Marks workflow termination nodes (handle_resume_making, mock_interview, job_search) linked to END.

Compilation: Sets categorize as the entry point and compiles the graph into an executable app for query processing.

In [ ]:
# Create the workflow graph
workflow = StateGraph(State)

# Add nodes for each state in the workflow
workflow.add_node("categorize", categorize)   
workflow.add_node("handle_learning_resource", handle_learning_resource)   
workflow.add_node("handle_resume_making", handle_resume_making)   
workflow.add_node("handle_interview_preparation", handle_interview_preparation)  
workflow.add_node("job_search", job_search)   
workflow.add_node("mock_interview", mock_interview)   
workflow.add_node("interview_topics_questions", interview_topics_questions)   
workflow.add_node("tutorial_agent", tutorial_agent)   
workflow.add_node("ask_query_bot", ask_query_bot)   

workflow.add_edge(START, "categorize")

# Add conditional edges based on category routing function
workflow.add_conditional_edges(
    "categorize",
    route_query,
    {
        "handle_learning_resource": "handle_learning_resource",
        "handle_resume_making": "handle_resume_making",
        "handle_interview_preparation": "handle_interview_preparation",
        "job_search": "job_search"
    }
)

# Add conditional edges for further routing in interview preparation
workflow.add_conditional_edges(
    "handle_interview_preparation",
    route_interview,
    {
        "mock_interview": "mock_interview",
        "interview_topics_questions": "interview_topics_questions",
    }
)

# Add conditional edges for further routing in learning resources
workflow.add_conditional_edges(
    "handle_learning_resource",
    route_learning,
    {
        "tutorial_agent": "tutorial_agent",
        "ask_query_bot": "ask_query_bot",
    }
)

# Define edges that lead to the end of the workflow
workflow.add_edge("handle_resume_making", END)
workflow.add_edge("job_search", END)
workflow.add_edge("interview_topics_questions", END)
workflow.add_edge("mock_interview", END)
workflow.add_edge("ask_query_bot", END)
workflow.add_edge("tutorial_agent", END)

workflow.set_entry_point("categorize")
 
app = workflow.compile()


### Lets Visualize our graph

- **Displaying Workflow Graph**:
  - Generates a visual representation of the `app` workflow graph using Mermaid, which is then displayed as a PNG image.
  - The `MermaidDrawMethod.API` method is used to create the PNG, ensuring a clear, structured view of the workflow nodes and their connections.


In [ ]:
# Display the workflow graph as a PNG image using Mermaid
display(
    Image(
        app.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,  
        )
    )
)


### Final function to Processes a user query using the LangGraph workflow and returns a dictionary containing the query's category and response.

In [43]:
def run_user_query(query: str) -> Dict[str, str]:
    """Process a user query through the LangGraph workflow.
    
    Args:
        query (str): The user's query
        
    Returns:
        Dict[str, str]: A dictionary containing the query's category and response
    """
    results = app.invoke({"query": query})
    return {
        "category": results["category"],
        "response": results["response"]
    }


# ---------------------------Testing Different Scenarios------------------------------

## TEST CASE 1: Creating Tutorials

In [ ]:
query = "I want to learn Langchain and langgraph.With usage and concept. Also give coding example implementation for both.Create tutorial for this."
result = run_user_query(query)
result

## TEST CASE 2: Q&A session for Doubts

In [ ]:
query = "I am confused between Langgraph and CrewAI when to use what for Agent Creation?"
result = run_user_query(query)
print(result)

## TEST CASE 3: Interview Question Discussion

In [ ]:
query = "I want to discussion Interview question for Gen AI job roles."
result = run_user_query(query)
print(result)

## TEST CASE 4: Mock Interview with Evaluation Feedback

In [ ]:
query = "I need mock interview to practice."
result = run_user_query(query)
result

## TEST CASE 5: Resume Modification Based on Job Description

In [ ]:
query = "Can you help me to modify my resume based on job description"
result = run_user_query(query)
result

## TEST CASE 6: Resume Making

In [ ]:
query = "I want to make resume for Gen AI roles job."
result = run_user_query(query)
result

## TEST CASE 7: Job Search

In [ ]:
query = "I want to search jobs."

result = run_user_query(query)
result